# Composite Mess3: Hierarchical Belief State Geometry

**Extension of**: "Transformers Represent Belief State Geometry in their Residual Stream"
(Shai et al., NeurIPS 2024)

A slow-switching **driver** (ε=0.05, dwell ≈ 20 steps) selects between two Mess3
**transducer** variants (α=0.85 vs α=0.65). The transformer sees only token emissions.
It must simultaneously infer the driver regime AND the transducer state from the
token stream alone.

**Key predictions:**
- Full 6D belief geometry is linearly recoverable from the residual stream
- Transducer synchronizes fast (~3 tokens); driver synchronizes slowly (~15-30 tokens)
- Driver and transducer information may occupy different layers

**Runtime**: GPU required. ~40-60 min for 1M steps on a T4.

In [ ]:
!pip install -q transformer-lens 2>&1 | tail -1
import torch, sys
print(f'Python {sys.version.split()[0]}  |  PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import itertools, warnings, json, os, zipfile
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformer_lens import HookedTransformer, HookedTransformerConfig

warnings.filterwarnings('ignore')

# ── Process parameters ──
X_A, A_A = 0.05, 0.85      # Variant A (same as paper's Mess3)
X_B, A_B = 0.05, 0.65      # Variant B (more diffuse)
EPSILON = 0.05              # Driver switching probability (dwell ≈ 20 steps)
VOCAB_SIZE = 3
N_STATES = 6                # 2 driver × 3 transducer

# ── Transformer architecture (Appendix A.6, with extended context) ──
N_CTX = 50                  # Extended from 10 to capture driver dynamics
D_MODEL = 64
D_HEAD = 8
N_HEADS = 1
N_LAYERS = 4
D_MLP = 256

# ── Training ──
BATCH_SIZE = 64
LEARNING_RATE = 0.01
SEQUENCE_LEN = N_CTX + 1
NUM_STEPS = 1_000_000
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Analysis ──
ANALYSIS_SEQS = 50_000      # Sampled sequences for analysis (exhaustive impossible at 3^50)

print(f'Device: {DEVICE}  |  Steps: {NUM_STEPS:,}  |  N_CTX: {N_CTX}')

# ── Figure output directory ──
FIG_DIR = 'composite_mess3_checkpoint/figures'
os.makedirs(FIG_DIR, exist_ok=True)

## 1. Define the Composite Process

Block transition matrix: current driver state determines transducer dynamics.
States: (D_A,S₁), (D_A,S₂), (D_A,S₃), (D_B,S₁), (D_B,S₂), (D_B,S₃).

In [ ]:
def mess3(x, a):
    b, y = (1 - a) / 2, 1 - 2 * x
    ay, bx, by, ax = a*y, b*x, b*y, a*x
    return np.array([
        [[ay,bx,bx],[ax,by,bx],[ax,bx,by]],
        [[by,ax,bx],[bx,ay,bx],[bx,ax,by]],
        [[by,bx,ax],[bx,by,ax],[bx,bx,ay]],
    ], dtype=np.float64)

def composite_mess3(x_a, a_a, x_b, a_b, epsilon):
    """Build (3, 6, 6) composite transition matrix."""
    t_a = mess3(x_a, a_a)
    t_b = mess3(x_b, a_b)
    T = np.zeros((3, 6, 6), dtype=np.float64)
    for y in range(3):
        T[y, :3, :3] = (1 - epsilon) * t_a[y]   # D_A → D_A, use T_A
        T[y, :3, 3:] = epsilon * t_a[y]           # D_A → D_B, use T_A
        T[y, 3:, :3] = epsilon * t_b[y]           # D_B → D_A, use T_B
        T[y, 3:, 3:] = (1 - epsilon) * t_b[y]   # D_B → D_B, use T_B
    return T

tmats = composite_mess3(X_A, A_A, X_B, A_B, EPSILON)

# ── Verify ──
T_net = tmats.sum(axis=0)
row_sums = T_net.sum(axis=1)
assert np.allclose(row_sums, 1.0, atol=1e-10), f'Row sums: {row_sums}'
assert np.all(tmats >= 0) and np.all(tmats <= 1)

eigvals, eigvecs = np.linalg.eig(T_net.T)
stationary = eigvecs[:, np.isclose(eigvals, 1)].real.squeeze()
stationary = stationary / stationary.sum()

print(f'Shape: {tmats.shape}  |  States: {N_STATES}  |  Vocab: {VOCAB_SIZE}')
print(f'Stationary: {stationary.round(4)}')
print(f'Driver marginal: P(D_A)={stationary[:3].sum():.4f}, P(D_B)={stationary[3:].sum():.4f}')
print(f'Row sums: {row_sums.round(6)}')
print(f'PASS: Composite transition matrix verified.')

## 2. Ground-Truth Belief Geometry

The MSP of this 6-state HMM should show two fractal clusters (one per driver
regime) connected by a bridge of uncertain-driver points.

In [ ]:
def build_msp_beliefs(tmats, init_state, max_depth):
    beliefs = [init_state.copy()]
    queue = [(init_state.copy(), 0)]
    while queue:
        state, depth = queue.pop(0)
        if depth >= max_depth:
            continue
        for x in range(tmats.shape[0]):
            raw = state @ tmats[x]
            norm = raw.sum()
            if norm > 1e-15:
                child = raw / norm
                beliefs.append(child)
                queue.append((child, depth + 1))
    return np.array(beliefs)

msp_beliefs = build_msp_beliefs(tmats, stationary, 7)
print(f'MSP nodes (depth 7): {len(msp_beliefs):,}')

# Driver marginal for each belief state
theta_gt = msp_beliefs[:, :3].sum(axis=1)

# PCA for visualization (6D → 2D)
from numpy.linalg import eigh
centered = msp_beliefs - msp_beliefs.mean(axis=0)
cov = centered.T @ centered / len(centered)
eigvals_pca, eigvecs_pca = eigh(cov)
order = np.argsort(eigvals_pca)[::-1]
pcs = eigvecs_pca[:, order[:2]]
proj = centered @ pcs

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Color by driver marginal theta
sc = axes[0].scatter(proj[:, 0], proj[:, 1], c=theta_gt, cmap='coolwarm', s=1, alpha=0.5)
plt.colorbar(sc, ax=axes[0], label=r'$\theta$ = P(driver=A)')
axes[0].set_title('Ground-Truth Belief Geometry\n(PCA of 5-simplex, colored by driver)')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2'); axes[0].set_aspect('equal')

# Color by belief state as RGB (first 3 components = variant A)
rgb = np.clip(np.column_stack([
    msp_beliefs[:, 0] + msp_beliefs[:, 3],  # total S1 mass
    msp_beliefs[:, 1] + msp_beliefs[:, 4],  # total S2 mass
    msp_beliefs[:, 2] + msp_beliefs[:, 5],  # total S3 mass
]), 0, 1)
axes[1].scatter(proj[:, 0], proj[:, 1], c=rgb, s=1, alpha=0.5)
axes[1].set_title('Ground-Truth Belief Geometry\n(colored by transducer marginal RGB)')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2'); axes[1].set_aspect('equal')

plt.tight_layout(); plt.show()

## 3. Initialize Transformer

Same architecture as the paper (Appendix A.6) but with N_CTX=50.

In [ ]:
torch.manual_seed(SEED)
try:
    tl_cfg = HookedTransformerConfig(
        n_ctx=N_CTX, d_model=D_MODEL, d_head=D_HEAD, n_heads=N_HEADS,
        n_layers=N_LAYERS, d_mlp=D_MLP, d_vocab=VOCAB_SIZE,
        act_fn='relu', normalization_type='LN', device=DEVICE, seed=SEED,
    )
    model = HookedTransformer(tl_cfg)
    actual_n_heads = N_HEADS
except Exception as e:
    print(f'n_heads=1 failed ({e}); using n_heads=8.')
    actual_n_heads = 8
    tl_cfg = HookedTransformerConfig(
        n_ctx=N_CTX, d_model=D_MODEL, d_head=D_HEAD, n_heads=actual_n_heads,
        n_layers=N_LAYERS, d_mlp=D_MLP, d_vocab=VOCAB_SIZE,
        act_fn='relu', normalization_type='LN', device=DEVICE, seed=SEED,
    )
    model = HookedTransformer(tl_cfg)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}  |  n_heads={actual_n_heads}  |  N_CTX={N_CTX}')

In [ ]:
# ── Data generation and belief computation functions ──
# These are needed by both training and analysis. Always run this cell.

def generate_batch(rng_obj):
    """Generate a batch of sequences from the composite HMM. Vectorized over batch."""
    states = np.tile(stationary, (BATCH_SIZE, 1))
    tokens = np.zeros((BATCH_SIZE, SEQUENCE_LEN), dtype=np.int64)
    for t in range(SEQUENCE_LEN):
        obs_probs = np.einsum('bi,xij->bx', states, tmats)
        obs_probs = np.maximum(obs_probs, 0)
        obs_probs /= obs_probs.sum(axis=1, keepdims=True)
        cumprobs = obs_probs.cumsum(axis=1)
        u = rng_obj.random(BATCH_SIZE)[:, None]
        tokens[:, t] = (u >= cumprobs).sum(axis=1)
        states = np.einsum('bi,bij->bj', states, tmats[tokens[:, t]])
        states /= states.sum(axis=1, keepdims=True)
    return (torch.tensor(tokens[:, :-1], device=DEVICE),
            torch.tensor(tokens[:, 1:], device=DEVICE, dtype=torch.long))

def generate_sequences(rng_obj, n_seqs, seq_len):
    """Generate n_seqs sequences of given length. Vectorized over batch."""
    states = np.tile(stationary, (n_seqs, 1))
    seqs = np.zeros((n_seqs, seq_len), dtype=np.int64)
    for t in range(seq_len):
        obs_probs = np.einsum('bi,xij->bx', states, tmats)
        obs_probs = np.maximum(obs_probs, 0)
        obs_probs /= obs_probs.sum(axis=1, keepdims=True)
        cumprobs = obs_probs.cumsum(axis=1)
        u = rng_obj.random(n_seqs)[:, None]
        seqs[:, t] = (u >= cumprobs).sum(axis=1)
        states = np.einsum('bi,bij->bj', states, tmats[seqs[:, t]])
        states /= states.sum(axis=1, keepdims=True)
    return seqs

def compute_beliefs_for_sequences(seqs_np):
    """Compute 6D belief states at every position. Vectorized over batch."""
    n, seq_len = seqs_np.shape
    states = np.tile(stationary, (n, 1))
    beliefs = np.zeros((n, seq_len, N_STATES), dtype=np.float32)
    for t in range(seq_len):
        states = np.einsum('bi,bij->bj', states, tmats[seqs_np[:, t]])
        states /= states.sum(axis=1, keepdims=True)
        beliefs[:, t] = states
    return beliefs

all_layer_keys = [f'blocks.{l}.hook_resid_post' for l in range(N_LAYERS)]

def run_checkpoint_analysis(n_seqs=5000):
    model.eval()
    seqs = generate_sequences(np.random.default_rng(SEED + 777), n_seqs, N_CTX)
    beliefs = compute_beliefs_for_sequences(seqs)
    beliefs_flat = beliefs.reshape(-1, N_STATES)
    n_data_ca = beliefs_flat.shape[0]

    seqs_t = torch.tensor(seqs, dtype=torch.long)
    acts_list = []
    with torch.no_grad():
        for s in range(0, n_seqs, 2048):
            e = min(s + 2048, n_seqs)
            _, cache = model.run_with_cache(seqs_t[s:e].to(DEVICE))
            acts_list.append(cache[all_layer_keys[-1]].detach().cpu().numpy())
            del cache
    acts_flat = np.concatenate(acts_list, axis=0).reshape(-1, D_MODEL)

    X = np.column_stack([np.ones(n_data_ca), acts_flat]).astype(np.float64)
    Y = beliefs_flat.astype(np.float64)
    beta, _, _, _ = np.linalg.lstsq(X, Y, rcond=None)
    resid = X @ beta - Y
    mse = float(np.mean(np.sum(resid**2, axis=1)))
    ss_res = np.sum(resid**2); ss_tot = np.sum((Y - Y.mean(axis=0))**2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return {'paper_mse': mse, 'r2': r2}

print(f'Defined: generate_batch, generate_sequences, compute_beliefs_for_sequences, run_checkpoint_analysis')
print(f'Layer keys: {all_layer_keys}')

## 4. Train OR Load Checkpoint

**Always run the helpers cell above first** (data generation functions).

Then choose one:
- **Option A**: Run the training cell (~1 hour on T4 GPU)
- **Option B**: Upload `composite_mess3_checkpoint.zip` and run the load cell (instant)

In [ ]:
# ── OPTION B: Load from checkpoint (skip training) ──
# Upload composite_mess3_checkpoint.zip to Colab first, then run this cell.
# If you already trained in this session, skip this cell.

import zipfile

zip_path = 'composite_mess3_checkpoint.zip'
save_dir = 'composite_mess3_checkpoint'

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('.')
    print(f'Extracted checkpoint to {save_dir}/')
elif not os.path.exists(save_dir):
    print('ERROR: No checkpoint found. Either:')
    print('  1. Upload composite_mess3_checkpoint.zip to Colab')
    print('  2. Or run the training cell instead')
    raise FileNotFoundError('No checkpoint available')

# Load model weights
model.load_state_dict(torch.load(f'{save_dir}/model_weights.pt', map_location=DEVICE))
model.eval()
print('Model weights loaded.')

# Load training history
th = np.load(f'{save_dir}/training_history.npz')
losses = list(th['losses'])
checkpoint_results = {}
for step, mse, r2 in zip(th['checkpoint_steps'], th['checkpoint_mses'], th['checkpoint_r2s']):
    checkpoint_results[int(step)] = {'paper_mse': float(mse), 'r2': float(r2)}
print(f'Training history: {len(losses):,} steps, final loss={losses[-1]:.4f}')

# Load config
with open(f'{save_dir}/config.json') as f:
    saved_config = json.load(f)
print(f'Config: {saved_config["num_steps_trained"]:,} steps trained')
print(f'  Full belief R²={saved_config["full_belief_r2_concat"]:.4f}')
print(f'  Driver R²={saved_config["driver_r2_concat"]:.4f}')
print(f'  Transducer R²={saved_config["transducer_r2_concat"]:.4f}')

print('\n✓ Checkpoint loaded. Skip the training cell, run loss-plot and analysis.')


In [ ]:
# ── OPTION A: Train from scratch (skip if loading checkpoint) ──
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)
rng = np.random.default_rng(SEED)

checkpoints = sorted(set(
    [0, 10_000, 50_000, 100_000, 250_000, 500_000, 750_000, NUM_STEPS]
))
checkpoints = [c for c in checkpoints if c <= NUM_STEPS]
checkpoint_results = {}
losses = []
model.train()

pbar = tqdm(range(NUM_STEPS + 1), desc='Training')
for step in pbar:
    if step in checkpoints:
        sc = run_checkpoint_analysis()
        checkpoint_results[step] = sc
        pbar.set_postfix(mse=f"{sc['paper_mse']:.5f}", r2=f"{sc['r2']:.4f}")
        model.train()
    if step == 0:
        continue
    inputs, labels = generate_batch(rng)
    outputs = model(inputs)
    loss = loss_fn(outputs.reshape(-1, VOCAB_SIZE), labels.reshape(-1))
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    losses.append(loss.item())
    if step % 5000 == 0:
        pbar.set_postfix(loss=f'{loss.item():.4f}')

print(f'\nFinal loss: {losses[-1]:.4f}  (expected entropy rate: ~0.929)')
for s in sorted(checkpoint_results):
    r = checkpoint_results[s]
    print(f"  Step {s:>9,}: MSE={r['paper_mse']:.6f}  R2={r['r2']:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.plot(losses, lw=0.2, alpha=0.3, label='per-step')
w = min(2000, len(losses) // 5) if len(losses) > 10 else 1
if w > 1:
    sm = np.convolve(losses, np.ones(w)/w, mode='valid')
    ax.plot(np.arange(w-1, w-1+len(sm)), sm, lw=1.2, label=f'{w}-step avg')
ax.axhline(0.929, color='red', ls='--', lw=0.8, label='Entropy rate (~0.929)')
ax.set_xlabel('Step'); ax.set_ylabel('Loss'); ax.set_title('Training Loss'); ax.legend()

ax = axes[1]
steps = sorted(checkpoint_results)
mses = [checkpoint_results[s]['paper_mse'] for s in steps]
ax.plot(steps, mses, 'o-', ms=5)
ax.set_xlabel('Step'); ax.set_ylabel('MSE (full belief)'); ax.set_title('Belief Geometry MSE')
ax.set_yscale('log')
for s, m in zip(steps, mses):
    ax.annotate(f'{m:.4f}', (s, m), textcoords='offset points', xytext=(5, 5), fontsize=7)

plt.tight_layout(); plt.savefig(f'{FIG_DIR}/fig_training.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Activation Analysis — Probes & Decompositions

**Probe** (Full belief η ∈ ℝ⁶): per-layer + concatenated. This is the primary result.

**Decompositions** (derived from full belief, not independent evidence):
- Driver marginal θ = η₁+η₂+η₃ ∈ ℝ¹ — a linear projection of η
- Transducer marginal μ = η[:3]+η[3:] ∈ ℝ³ — a linear projection of η
- Both are mechanically recoverable from any linear map that recovers η

**Timescale**: R² vs context position for driver and transducer decompositions.

*Note: θ and μ are linear functions of η, so their R² is guaranteed ≥ the full-belief R².
The per-layer pattern is informative as a decomposition, not as independent evidence.*

In [ ]:
# ── Generate analysis data (sampled, not exhaustive) ──
print(f'Generating {ANALYSIS_SEQS:,} sequences (vectorized)...')
analysis_seqs = generate_sequences(np.random.default_rng(SEED + 999), ANALYSIS_SEQS, N_CTX)

print('Computing belief states (vectorized)...')
analysis_beliefs = compute_beliefs_for_sequences(analysis_seqs)
# Shape: (ANALYSIS_SEQS, N_CTX, 6)

# Derive probe targets
eta_flat = analysis_beliefs.reshape(-1, N_STATES)                      # Full belief (6D)
theta_flat = analysis_beliefs[:, :, :3].sum(axis=-1).reshape(-1, 1)    # Driver marginal (1D)
mu_flat = (analysis_beliefs[:, :, :3].sum(axis=-1, keepdims=True) *    # Transducer marginal (3D)
    (analysis_beliefs[:, :, :3] / np.clip(analysis_beliefs[:, :, :3].sum(axis=-1, keepdims=True), 1e-10, None)) +
    analysis_beliefs[:, :, 3:].sum(axis=-1, keepdims=True) *
    (analysis_beliefs[:, :, 3:] / np.clip(analysis_beliefs[:, :, 3:].sum(axis=-1, keepdims=True), 1e-10, None))
).reshape(-1, 3)

n_data = eta_flat.shape[0]
print(f'Data points: {n_data:,}  |  Targets: eta(6D), theta(1D), mu(3D)')

# ── Extract activations from ALL layers ──
print('Extracting activations from all layers...')
model.eval()
seqs_t = torch.tensor(analysis_seqs, dtype=torch.long)
layer_acts_all = {k: [] for k in all_layer_keys}
with torch.no_grad():
    for s in range(0, ANALYSIS_SEQS, 2048):
        e = min(s + 2048, ANALYSIS_SEQS)
        _, cache = model.run_with_cache(seqs_t[s:e].to(DEVICE))
        for k in all_layer_keys:
            layer_acts_all[k].append(cache[k].detach().cpu().numpy())
        del cache
        if (s // 2048) % 5 == 0:
            print(f'  {s:,}/{ANALYSIS_SEQS:,}')

layer_acts_flat = {k: np.concatenate(v, axis=0).reshape(-1, D_MODEL) for k, v in layer_acts_all.items()}
concat_acts = np.concatenate([layer_acts_flat[k] for k in all_layer_keys], axis=1)
print(f'Per-layer: {D_MODEL}D  |  Concatenated: {concat_acts.shape[1]}D')

In [ ]:
# ── Run all probes: per-layer + concatenated ──
def run_probe(X_data, Y_data, label):
    X = np.column_stack([np.ones(X_data.shape[0]), X_data]).astype(np.float64)
    Y = Y_data.astype(np.float64)
    beta, _, _, _ = np.linalg.lstsq(X, Y, rcond=None)
    resid = X @ beta - Y
    mse = float(np.mean(np.sum(resid**2, axis=1)))
    ss_res = np.sum(resid**2); ss_tot = np.sum((Y - Y.mean(axis=0))**2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return {'mse': mse, 'r2': r2, 'label': label}

probes = {
    'full_belief': eta_flat,
    'driver': theta_flat,
    'transducer': mu_flat,
}

results = {}
for probe_name, target in probes.items():
    results[probe_name] = {}
    # Per-layer
    for k in all_layer_keys:
        layer_label = k.replace('blocks.', 'L').replace('.hook_resid_post', '')
        r = run_probe(layer_acts_flat[k], target, layer_label)
        results[probe_name][layer_label] = r
    # Concatenated
    r = run_probe(concat_acts, target, 'Concat')
    results[probe_name]['Concat'] = r

# ── Display results table ──
print('=' * 70)
print(f'{"Probe":<18} {"Layer":<8} {"R²":>8} {"MSE":>12}')
print('-' * 70)
for probe_name in probes:
    for layer_label in [f'L{l}' for l in range(N_LAYERS)] + ['Concat']:
        r = results[probe_name][layer_label]
        print(f'{probe_name:<18} {layer_label:<8} {r["r2"]:>8.4f} {r["mse"]:>12.6f}')
    print()

# ── Bar chart (Figure 7E analogue) ──
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, probe_name in zip(axes, probes):
    labels_b = [f'L{l}' for l in range(N_LAYERS)] + ['Concat']
    r2s = [results[probe_name][l]['r2'] for l in labels_b]
    colors_b = ['#90CAF9'] * N_LAYERS + ['#2196F3']
    ax.bar(labels_b, r2s, color=colors_b, alpha=0.8)
    ax.set_ylabel('R²'); ax.set_title(f'{probe_name}')
    ax.set_ylim(0, 1.05)
    for i, v in enumerate(r2s):
        ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=8)
plt.suptitle('Per-Layer R² for Each Probe', fontsize=14)
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/fig_probes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Timescale separation: corrected (Prediction P4) ──
# Original per-position R² was misleading: at position 0, the driver marginal θ
# is identical for ALL sequences (θ=0.5 exactly, zero variance), making R²
# undefined (0/0). At positions 1-2, variance is near-zero, inflating R².
# Fix: fit ONE regression on ALL positions, evaluate MSE per position.
# MSE is the primary metric (not affected by target variance normalization).

print('Computing corrected timescale separation...')

final_acts_3d = np.concatenate(layer_acts_all[all_layer_keys[-1]], axis=0)  # (50K, 50, 64)

# Also build concatenated activations per position for comparison
all_layer_3d = {}
for k in all_layer_keys:
    all_layer_3d[k] = np.concatenate(layer_acts_all[k], axis=0)  # (50K, 50, 64)

concat_acts_3d = np.concatenate([all_layer_3d[k] for k in all_layer_keys], axis=-1)  # (50K, 50, 256)
concat_dim_ts = concat_acts_3d.shape[-1]

theta_3d = analysis_beliefs[:, :, :3].sum(axis=-1)  # (50K, 50)
theta_flat_ts = theta_3d.reshape(-1, 1)

def transducer_marginal(beliefs):
    theta = beliefs[..., :3].sum(axis=-1, keepdims=True)
    beta_a = beliefs[..., :3] / np.clip(theta, 1e-10, None)
    beta_b = beliefs[..., 3:] / np.clip(1 - theta, 1e-10, None)
    return theta * beta_a + (1 - theta) * beta_b

mu_3d = transducer_marginal(analysis_beliefs)  # (50K, 50, 3)
mu_flat_ts = mu_3d.reshape(-1, 3)

# ── Fit single regressions on ALL positions: final layer AND concatenated ──
# Final layer
acts_flat_final = final_acts_3d.reshape(-1, D_MODEL)
X_final = np.column_stack([np.ones(acts_flat_final.shape[0]), acts_flat_final]).astype(np.float64)
beta_driver_final, _, _, _ = np.linalg.lstsq(X_final, theta_flat_ts.astype(np.float64), rcond=None)
beta_trans_final, _, _, _ = np.linalg.lstsq(X_final, mu_flat_ts.astype(np.float64), rcond=None)

# Concatenated
acts_flat_concat = concat_acts_3d.reshape(-1, concat_dim_ts)
X_concat = np.column_stack([np.ones(acts_flat_concat.shape[0]), acts_flat_concat]).astype(np.float64)
beta_driver_concat, _, _, _ = np.linalg.lstsq(X_concat, theta_flat_ts.astype(np.float64), rcond=None)
beta_trans_concat, _, _, _ = np.linalg.lstsq(X_concat, mu_flat_ts.astype(np.float64), rcond=None)

# ── Evaluate per position for both probes ──
results_ts = {}
for probe_label, acts_3d, beta_d, beta_t in [
    ('L3', final_acts_3d, beta_driver_final, beta_trans_final),
    ('Concat', concat_acts_3d, beta_driver_concat, beta_trans_concat),
]:
    d_mse = np.zeros(N_CTX)
    d_r2 = np.full(N_CTX, np.nan)
    t_mse = np.zeros(N_CTX)
    t_r2 = np.full(N_CTX, np.nan)
    t_std = np.zeros(N_CTX)
    dim = acts_3d.shape[-1]

    for t in range(N_CTX):
        X_t = np.column_stack([np.ones(ANALYSIS_SEQS), acts_3d[:, t]]).astype(np.float64)

        y_d = theta_3d[:, t:t+1].astype(np.float64)
        pred_d = X_t @ beta_d
        resid_d = pred_d - y_d
        d_mse[t] = float(np.mean(resid_d**2))
        ss_tot_d = float(np.sum((y_d - y_d.mean())**2))
        if ss_tot_d > 1e-10:
            d_r2[t] = 1 - np.sum(resid_d**2) / ss_tot_d

        y_t = mu_3d[:, t].astype(np.float64)
        pred_t = X_t @ beta_t
        resid_t = pred_t - y_t
        t_mse[t] = float(np.mean(np.sum(resid_t**2, axis=1)))
        ss_tot_t = float(np.sum((y_t - y_t.mean(axis=0))**2))
        if ss_tot_t > 1e-10:
            t_r2[t] = 1 - np.sum(resid_t**2) / ss_tot_t

        t_std[t] = float(y_d.std())

    results_ts[probe_label] = {
        'driver_mse': d_mse, 'driver_r2': d_r2,
        'trans_mse': t_mse, 'trans_r2': t_r2,
        'theta_std': t_std,
    }

# Use L3 theta_std (same for both)
theta_std_by_pos = results_ts['L3']['theta_std']
reliable = theta_std_by_pos > 0.05

# ── Four-panel plot: MSE (L3 + Concat) + R² (L3) + variability ──
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for panel_idx, (label, color_d, color_t, ls) in enumerate([
    ('L3 (final layer)', '#EF9F27', '#1D9E75', '-'),
    ('Concat (all layers)', '#D84315', '#00695C', '--'),
]):
    ax = axes[panel_idx]
    r = results_ts[label.split(' ')[0]]
    ax.plot(range(N_CTX), r['trans_mse'], 'o-', ms=2, lw=1.5, color=color_t,
            label=f'Transducer')
    ax.plot(range(N_CTX), r['driver_mse'], 's-', ms=2, lw=1.5, color=color_d,
            label=f'Driver')
    ax.set_xlabel('Context position t')
    ax.set_ylabel('MSE')
    ax.set_title(f'MSE vs Position — {label}')
    ax.legend(fontsize=8)

ax = axes[2]
pos_range = np.arange(N_CTX)
for label, color_d, color_t, ls in [
    ('L3', '#EF9F27', '#1D9E75', '-'),
    ('Concat', '#D84315', '#00695C', '--'),
]:
    r = results_ts[label]
    if np.any(reliable):
        ax.plot(pos_range[reliable], r['trans_r2'][reliable],
                f'o{ls}', ms=2, lw=1.5, color=color_t, label=f'Trans ({label})')
        ax.plot(pos_range[reliable], r['driver_r2'][reliable],
                f's{ls}', ms=2, lw=1.5, color=color_d, label=f'Driver ({label})')
ax.axhline(0.9, color='gray', ls=':', lw=0.5)
ax.axhline(0.8, color='gray', ls=':', lw=0.5)
ax.set_xlabel('Context position t')
ax.set_ylabel(r'R$^2$')
ax.set_title(r'R$^2$ vs Position (std > 0.05)')
ax.legend(fontsize=7)
ax.set_ylim(-0.05, 1.05)

ax = axes[3]
ax.plot(range(N_CTX), theta_std_by_pos, 'k-o', ms=2, lw=1.5)
ax.axhline(0.05, color='red', ls='--', lw=0.8, label='Reliability threshold')
ax.set_xlabel('Context position t')
ax.set_ylabel(r'Std($\theta$)')
ax.set_title(r'Driver Target Variability')
ax.legend()

plt.suptitle('Timescale Separation (L3 vs Concatenated)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_timescale.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Summary ──
for label in ['L3', 'Concat']:
    r = results_ts[label]
    print(f'\n=== {label} ===')
    for t in [1, 5, 10, 15, 20, 30, 49]:
        if t < N_CTX and reliable[t]:
            print(f'  pos {t:2d}: driver MSE={r["driver_mse"][t]:.6f} R²={r["driver_r2"][t]:.4f}  |  trans MSE={r["trans_mse"][t]:.6f} R²={r["trans_r2"][t]:.4f}')

# Save updated arrays
driver_mse_by_pos = results_ts['L3']['driver_mse']
driver_r2_by_pos = results_ts['L3']['driver_r2']
transducer_mse_by_pos = results_ts['L3']['trans_mse']
transducer_r2_by_pos = results_ts['L3']['trans_r2']


## 6. Controls

Cross-validation (20% train / 80% test) and shuffle, using the full belief probe.

In [ ]:
# ── Controls on the concatenated full-belief probe ──
N_TRIALS = 100
ctrl_rng = np.random.default_rng(SEED)

# ── Control 1: Untrained model baseline ──
print('Running untrained-model baseline...')
torch.manual_seed(SEED + 12345)
try:
    untrained_cfg = HookedTransformerConfig(
        n_ctx=N_CTX, d_model=D_MODEL, d_head=D_HEAD, n_heads=N_HEADS,
        n_layers=N_LAYERS, d_mlp=D_MLP, d_vocab=VOCAB_SIZE,
        act_fn='relu', normalization_type='LN', device=DEVICE, seed=SEED+12345,
    )
    untrained_model = HookedTransformer(untrained_cfg)
except Exception:
    untrained_cfg = HookedTransformerConfig(
        n_ctx=N_CTX, d_model=D_MODEL, d_head=D_HEAD, n_heads=8,
        n_layers=N_LAYERS, d_mlp=D_MLP, d_vocab=VOCAB_SIZE,
        act_fn='relu', normalization_type='LN', device=DEVICE, seed=SEED+12345,
    )
    untrained_model = HookedTransformer(untrained_cfg)
untrained_model.eval()

seqs_t = torch.tensor(analysis_seqs, dtype=torch.long)
untrained_layer_acts = {k: [] for k in all_layer_keys}
with torch.no_grad():
    for s in range(0, ANALYSIS_SEQS, 2048):
        e = min(s + 2048, ANALYSIS_SEQS)
        _, cache = untrained_model.run_with_cache(seqs_t[s:e].to(DEVICE))
        for k in all_layer_keys:
            untrained_layer_acts[k].append(cache[k].detach().cpu().numpy())
        del cache

untrained_concat = np.concatenate(
    [np.concatenate(v, axis=0).reshape(-1, D_MODEL) for v in untrained_layer_acts.values()],
    axis=1
)
del untrained_model, untrained_layer_acts
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

X_untrained = np.column_stack([np.ones(n_data), untrained_concat]).astype(np.float64)
beta_un, _, _, _ = np.linalg.lstsq(X_untrained, eta_flat.astype(np.float64), rcond=None)
resid_un = X_untrained @ beta_un - eta_flat
untrained_mse = float(np.mean(np.sum(resid_un**2, axis=1)))
ss_res_un = np.sum(resid_un**2)
ss_tot_un = np.sum((eta_flat - eta_flat.mean(axis=0))**2)
untrained_r2 = 1 - ss_res_un / ss_tot_un
del X_untrained, untrained_concat
print(f'Untrained model: MSE={untrained_mse:.6f}, R²={untrained_r2:.4f}')

# ── Control 2 (PRIMARY): Sequence-level CV (20% train / 80% test) ──
# Holds out entire sequences, preventing within-sequence leakage.
seq_cv_test, seq_cv_train = [], []
n_seqs = ANALYSIS_SEQS
for _ in tqdm(range(N_TRIALS), desc='Seq-CV (primary)'):
    seq_idx = ctrl_rng.permutation(n_seqs)
    n_train_seqs = int(0.2 * n_seqs)
    train_seqs = seq_idx[:n_train_seqs]
    test_seqs = seq_idx[n_train_seqs:]
    tr_i = np.concatenate([np.arange(s * N_CTX, (s + 1) * N_CTX) for s in train_seqs])
    te_i = np.concatenate([np.arange(s * N_CTX, (s + 1) * N_CTX) for s in test_seqs])
    X_tr = np.column_stack([np.ones(len(tr_i)), concat_acts[tr_i]]).astype(np.float64)
    beta, _, _, _ = np.linalg.lstsq(X_tr, eta_flat[tr_i].astype(np.float64), rcond=None)
    X_te = np.column_stack([np.ones(len(te_i)), concat_acts[te_i]]).astype(np.float64)
    te_resid = X_te @ beta - eta_flat[te_i]
    seq_cv_test.append(float(np.mean(np.sum(te_resid**2, axis=1))))
    tr_resid = X_tr @ beta - eta_flat[tr_i]
    seq_cv_train.append(float(np.mean(np.sum(tr_resid**2, axis=1))))

# ── Control 3: Point-level CV (for comparison — has within-sequence leakage) ──
pt_cv_test, pt_cv_train = [], []
for _ in tqdm(range(N_TRIALS), desc='Point-CV (comparison)'):
    idx = ctrl_rng.permutation(n_data)
    nt = int(0.2 * n_data)
    tr_i, te_i = idx[:nt], idx[nt:]
    X_tr = np.column_stack([np.ones(nt), concat_acts[tr_i]]).astype(np.float64)
    beta, _, _, _ = np.linalg.lstsq(X_tr, eta_flat[tr_i].astype(np.float64), rcond=None)
    X_te = np.column_stack([np.ones(len(te_i)), concat_acts[te_i]]).astype(np.float64)
    te_resid = X_te @ beta - eta_flat[te_i]
    pt_cv_test.append(float(np.mean(np.sum(te_resid**2, axis=1))))
    tr_resid = X_tr @ beta - eta_flat[tr_i]
    pt_cv_train.append(float(np.mean(np.sum(tr_resid**2, axis=1))))

# ── Control 4: Shuffle ──
shuffle_mses = []
for _ in tqdm(range(N_TRIALS), desc='Shuffle'):
    perm = ctrl_rng.permutation(n_data)
    X = np.column_stack([np.ones(n_data), concat_acts]).astype(np.float64)
    beta, _, _, _ = np.linalg.lstsq(X, eta_flat[perm].astype(np.float64), rcond=None)
    resid = X @ beta - eta_flat[perm]
    shuffle_mses.append(float(np.mean(np.sum(resid**2, axis=1))))

# ── Control 5: Temporal split — train on first half, test on second half ──
# Tests whether the linear map generalizes across context positions.
mid = N_CTX // 2
first_half_idx = np.concatenate([np.arange(s * N_CTX, s * N_CTX + mid) for s in range(n_seqs)])
second_half_idx = np.concatenate([np.arange(s * N_CTX + mid, (s + 1) * N_CTX) for s in range(n_seqs)])

X_first = np.column_stack([np.ones(len(first_half_idx)), concat_acts[first_half_idx]]).astype(np.float64)
beta_t, _, _, _ = np.linalg.lstsq(X_first, eta_flat[first_half_idx].astype(np.float64), rcond=None)
X_second = np.column_stack([np.ones(len(second_half_idx)), concat_acts[second_half_idx]]).astype(np.float64)
tsplit_resid = X_second @ beta_t - eta_flat[second_half_idx]
tsplit_mse = float(np.mean(np.sum(tsplit_resid**2, axis=1)))

# Reverse direction
X_second2 = np.column_stack([np.ones(len(second_half_idx)), concat_acts[second_half_idx]]).astype(np.float64)
beta_t2, _, _, _ = np.linalg.lstsq(X_second2, eta_flat[second_half_idx].astype(np.float64), rcond=None)
X_first2 = np.column_stack([np.ones(len(first_half_idx)), concat_acts[first_half_idx]]).astype(np.float64)
tsplit_resid2 = X_first2 @ beta_t2 - eta_flat[first_half_idx]
tsplit_mse_rev = float(np.mean(np.sum(tsplit_resid2**2, axis=1)))
tsplit_mse_avg = (tsplit_mse + tsplit_mse_rev) / 2

# ── Results ──
full_mse = results['full_belief']['Concat']['mse']
print('\n=== Controls (full belief, concatenated) ===')
print(f'Full MSE:              {full_mse:.6f}')
print(f'Untrained MSE:         {untrained_mse:.6f}  (R²={untrained_r2:.4f})')
print(f'Seq-CV test MSE:       {np.mean(seq_cv_test):.6f} +/- {np.std(seq_cv_test):.6f}  [PRIMARY]')
print(f'Point-CV test MSE:     {np.mean(pt_cv_test):.6f} +/- {np.std(pt_cv_test):.6f}  [has within-seq leakage]')
print(f'Temporal split MSE:    {tsplit_mse_avg:.6f}  (train first/second half, test other)')
print(f'Shuffle MSE:           {np.mean(shuffle_mses):.6f} +/- {np.std(shuffle_mses):.6f}')

sh_ratio = np.mean(shuffle_mses) / max(full_mse, 1e-10)
un_ratio = untrained_mse / max(full_mse, 1e-10)
seq_cv_ratio = np.mean(seq_cv_test) / max(full_mse, 1e-10)
tsplit_ratio = tsplit_mse_avg / max(full_mse, 1e-10)
print(f'\nUntrained/Full:        {un_ratio:.1f}x')
print(f'Shuffle/Full:          {sh_ratio:.1f}x')
print(f'Seq-CV test/Full:      {seq_cv_ratio:.2f}x  [primary — no within-seq leakage]')
print(f'Temporal split/Full:   {tsplit_ratio:.2f}x  [generalizes across positions?]')

# ── Plot ──
fig, ax = plt.subplots(figsize=(10, 4.5))
labels_c = ['Full\n(trained)', 'Untrained\nbaseline', 'Seq-CV\ntest (primary)',
            'Temporal\nsplit', 'Shuffle']
means_c = [full_mse, untrained_mse, np.mean(seq_cv_test), tsplit_mse_avg,
           np.mean(shuffle_mses)]
stds_c = [0, 0, np.std(seq_cv_test), abs(tsplit_mse - tsplit_mse_rev) / 2,
          np.std(shuffle_mses)]
colors_c = ['#2196F3', '#FF9800', '#9C27B0', '#00897B', '#F44336']
bars = ax.bar(labels_c, means_c, yerr=stds_c, capsize=5, color=colors_c, alpha=0.8)
ax.set_ylabel('MSE'); ax.set_title('Controls: Full Belief Probe (Concatenated)')
for bar, m in zip(bars, means_c):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(), f'{m:.4f}',
            ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/fig_controls.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Visualization

**Figure 5B/C analogue**: ground-truth belief geometry vs residual stream projection.
**Per-layer evolution**: how belief geometry builds through transformer layers.
**Subspace analysis**: are driver and transducer representations orthogonal?

In [ ]:
# ── Belief Geometry: Ground Truth vs Per-Layer Residual Stream ──

n_plot = min(100_000, n_data)
rng_viz = np.random.default_rng(42)
plot_idx = rng_viz.choice(n_data, n_plot, replace=False)

# Predicted beliefs from each layer + concatenated
layer_preds = {}
for k in all_layer_keys:
    lbl = k.replace('blocks.', 'L').replace('.hook_resid_post', '')
    X = np.column_stack([np.ones(n_data), layer_acts_flat[k]]).astype(np.float64)
    beta, _, _, _ = np.linalg.lstsq(X, eta_flat.astype(np.float64), rcond=None)
    layer_preds[lbl] = (X @ beta).astype(np.float32)

X_c = np.column_stack([np.ones(n_data), concat_acts]).astype(np.float64)
beta_c, _, _, _ = np.linalg.lstsq(X_c, eta_flat.astype(np.float64), rcond=None)
layer_preds['Concat'] = (X_c @ beta_c).astype(np.float32)

# PCA basis from ground truth beliefs
gt_sub = eta_flat[plot_idx]
gt_mean = gt_sub.mean(axis=0)
gt_centered = gt_sub - gt_mean
cov = gt_centered.T @ gt_centered / len(gt_centered)
eigvals_v, eigvecs_v = np.linalg.eigh(cov)
pcs = eigvecs_v[:, np.argsort(eigvals_v)[::-1][:2]]
proj_gt = gt_centered @ pcs
theta_sub = gt_sub[:, :3].sum(axis=1)

# ── 2×3 grid: Ground Truth + L0-L3 + Concat ──
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
panels = ['Ground Truth'] + [f'L{l}' for l in range(N_LAYERS)] + ['Concat']

for idx, lbl in enumerate(panels):
    row, col = divmod(idx, 3)
    ax = axes[row, col]
    if lbl == 'Ground Truth':
        proj = proj_gt
        title = r'Ground Truth $\eta$'
    else:
        pred_sub = layer_preds[lbl][plot_idx]
        proj = (pred_sub - gt_mean) @ pcs
        r2 = results['full_belief'][lbl]['r2']
        title = f'{lbl}  (R²={r2:.3f})'
    sc = ax.scatter(proj[:, 0], proj[:, 1], c=theta_sub, cmap='coolwarm',
                    s=0.2, alpha=0.3, rasterized=True)
    plt.colorbar(sc, ax=ax, label=r'$\theta$', shrink=0.8)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

plt.suptitle('Belief Geometry: Ground Truth vs Residual Stream (per layer)',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_perlayer_grid.png', dpi=150, bbox_inches='tight')
plt.show()

# ── MSP fractal vs Learned geometry (Figure 5B/C analogue) ──
# Recompute MSP if needed
try:
    _ = msp_beliefs.shape
except NameError:
    msp_beliefs = build_msp_beliefs(tmats, stationary, 7)

msp_mean = msp_beliefs.mean(axis=0)
msp_centered = msp_beliefs - msp_mean
cov_msp = msp_centered.T @ msp_centered / len(msp_centered)
eigvals_m, eigvecs_m = np.linalg.eigh(cov_msp)
pcs_msp = eigvecs_m[:, np.argsort(eigvals_m)[::-1][:2]]
proj_msp = msp_centered @ pcs_msp
theta_msp = msp_beliefs[:, :3].sum(axis=1)

# Project learned geometry onto MSP PCA basis
pred_concat_sub = layer_preds['Concat'][plot_idx]
proj_learned = (pred_concat_sub - msp_mean) @ pcs_msp

# Subsample learned geometry to match MSP density for fair comparison
n_msp = len(msp_beliefs)
learned_plot_idx = rng_viz.choice(len(proj_learned), min(n_msp, len(proj_learned)), replace=False)
theta_sub_matched = theta_sub[learned_plot_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

sc = axes[0].scatter(proj_msp[:, 0], proj_msp[:, 1], c=theta_msp, cmap='coolwarm',
                     s=2, alpha=0.6, vmin=0, vmax=1, rasterized=True)
plt.colorbar(sc, ax=axes[0], label=r'$\theta$ = P(driver=A)')
axes[0].set_title(f'MSP Fractal (depth 7, {len(msp_beliefs):,} nodes)')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].set_aspect('equal')

sc = axes[1].scatter(proj_learned[learned_plot_idx, 0], proj_learned[learned_plot_idx, 1],
                     c=theta_sub_matched, cmap='coolwarm',
                     s=2, alpha=0.6, vmin=0, vmax=1, rasterized=True)
plt.colorbar(sc, ax=axes[1], label=r'$\theta$ = P(driver=A)')
axes[1].set_title(f'Learned Geometry ({n_msp:,} points, matched density)')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].set_aspect('equal')

plt.suptitle("MSP Fractal vs Transformer's Learned Geometry", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_msp_vs_learned.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Subspace Analysis: Driver vs Transducer ──
# Key question: does the model encode driver and transducer in DISTINCT
# directions, or is this just a generic property of the encoding?
#
# Correct null: pick random linear projections of eta, regress them through
# the SAME activations, measure overlap. If ALL belief projections yield
# orthogonal weight vectors, then driver/transducer orthogonality is a
# property of the encoding geometry, not a specific organizational choice.
#
# Optimization: since every null target = eta @ w, the regression weights are
# B_eta @ w where B_eta = (X'X)^{-1} X' eta is precomputed once.

print('Subspace analysis (optimized)...')

X_sub = np.column_stack([np.ones(n_data), concat_acts]).astype(np.float64)
concat_dim = concat_acts.shape[1]

# ── Observed: driver and transducer regression weights ──
beta_d, _, _, _ = np.linalg.lstsq(X_sub, eta_flat[:, :3].sum(axis=1, keepdims=True).astype(np.float64), rcond=None)
driver_weights = beta_d[1:].squeeze()

beta_tr, _, _, _ = np.linalg.lstsq(X_sub, mu_flat.astype(np.float64), rcond=None)
trans_weights = beta_tr[1:]

driver_dir = driver_weights / np.linalg.norm(driver_weights)
obs_cos = []
for i in range(3):
    trans_dir = trans_weights[:, i] / np.linalg.norm(trans_weights[:, i])
    obs_cos.append(abs(np.dot(driver_dir, trans_dir)))

U_obs, _, _ = np.linalg.svd(trans_weights, full_matrices=False)
projection = U_obs @ (U_obs.T @ driver_weights)
obs_overlap = np.linalg.norm(projection) / np.linalg.norm(driver_weights)

# ── Precompute B_eta = (X'X)^{-1} X' eta  for the concat probe ──
# This lets us compute regression weights for ANY linear function of eta
# as just B_eta @ w, avoiding repeated lstsq calls.
print('Precomputing projection matrix...')
eta_f64 = eta_flat.astype(np.float64)
B_eta, _, _, _ = np.linalg.lstsq(X_sub, eta_f64, rcond=None)  # (257, 6)
B_eta_weights = B_eta[1:]  # (256, 6) -- skip intercept

# ── Null 1 (NAIVE): random directions in activation space ──
N_NULL = 10_000
null_rng = np.random.default_rng(42)
naive_overlap = []
for _ in range(N_NULL):
    rd = null_rng.standard_normal(concat_dim)
    rd = rd / np.linalg.norm(rd)
    rt = null_rng.standard_normal((concat_dim, 3))
    rt, _ = np.linalg.qr(rt)
    rt = rt[:, :3]
    proj_n = rt @ (rt.T @ rd)
    naive_overlap.append(np.linalg.norm(proj_n))
naive_overlap = np.array(naive_overlap)

# ── Null 2 (CORRECT): random linear projections of eta ──
# For target y = eta @ w, regression weights = B_eta_weights @ w
print('Computing belief-projection null (fast)...')
belief_overlap = []
for _ in range(N_NULL):
    w1 = null_rng.standard_normal(N_STATES)
    d1 = B_eta_weights @ w1  # (256,)

    W3 = null_rng.standard_normal((N_STATES, 3))
    d3 = B_eta_weights @ W3  # (256, 3)

    U_n, _, _ = np.linalg.svd(d3, full_matrices=False)
    proj_n = U_n @ (U_n.T @ d1)
    norm_d1 = np.linalg.norm(d1)
    belief_overlap.append(np.linalg.norm(proj_n) / max(norm_d1, 1e-15))
belief_overlap = np.array(belief_overlap)

# ── Results ──
naive_z = (obs_overlap - naive_overlap.mean()) / naive_overlap.std()
belief_z = (obs_overlap - belief_overlap.mean()) / belief_overlap.std()

print()
print('=== Subspace Overlap Analysis (256D concatenated) ===')
print(f'Observed overlap: {obs_overlap:.4f}')
print()
print(f'Naive null (random directions):')
print(f'  mean +/- std: {naive_overlap.mean():.4f} +/- {naive_overlap.std():.4f}')
print(f'  z-score: {naive_z:+.2f}')
print(f'  -> In 256D, everything is approximately orthogonal to everything.')
print(f'     This null is uninformative.')
print()
print(f'Belief-projection null (random linear functions of eta):')
print(f'  mean +/- std: {belief_overlap.mean():.4f} +/- {belief_overlap.std():.4f}')
print(f'  z-score: {belief_z:+.2f}')
if abs(belief_z) > 2:
    direction = 'MORE' if belief_z < -2 else 'LESS'
    print(f'  -> Driver and transducer are significantly {direction} orthogonal')
    print(f'     than random belief projections. The model specifically')
    print(f'     separates these quantities.')
else:
    print(f'  -> Driver/transducer overlap is TYPICAL for belief-space projections.')
    print(f'     The encoding maps eta linearly; driver and transducer happen to be')
    print(f'     approximately orthogonal IN BELIEF SPACE, and this is preserved.')

# ── Per-layer with belief null (also optimized) ──
print()
print('=== Per-layer (64D) ===')
naive_64 = []
for _ in range(N_NULL):
    rd = null_rng.standard_normal(D_MODEL)
    rd /= np.linalg.norm(rd)
    rt = null_rng.standard_normal((D_MODEL, 3))
    rt, _ = np.linalg.qr(rt)
    rt = rt[:, :3]
    proj_n = rt @ (rt.T @ rd)
    naive_64.append(np.linalg.norm(proj_n))
naive_64 = np.array(naive_64)
print(f'Naive null (64D) mean +/- std: {naive_64.mean():.4f} +/- {naive_64.std():.4f}')

for k in all_layer_keys:
    lbl = k.replace('blocks.', 'L').replace('.hook_resid_post', '')
    X_l = np.column_stack([np.ones(n_data), layer_acts_flat[k]]).astype(np.float64)

    b_d, _, _, _ = np.linalg.lstsq(X_l, eta_flat[:, :3].sum(axis=1, keepdims=True).astype(np.float64), rcond=None)
    b_t, _, _, _ = np.linalg.lstsq(X_l, mu_flat.astype(np.float64), rcond=None)
    d_w = b_d[1:].squeeze()
    t_w = b_t[1:]
    U_l, _, _ = np.linalg.svd(t_w, full_matrices=False)
    proj_l = U_l @ (U_l.T @ d_w)
    ov = np.linalg.norm(proj_l) / np.linalg.norm(d_w)

    B_l, _, _, _ = np.linalg.lstsq(X_l, eta_f64, rcond=None)
    B_l_w = B_l[1:]  # (64, 6)

    belief_ov_l = []
    for _ in range(N_NULL):
        w1 = null_rng.standard_normal(N_STATES)
        d1_ = B_l_w @ w1
        W3 = null_rng.standard_normal((N_STATES, 3))
        d3_ = B_l_w @ W3
        U_, _, _ = np.linalg.svd(d3_, full_matrices=False)
        p_ = U_ @ (U_.T @ d1_)
        belief_ov_l.append(np.linalg.norm(p_) / max(np.linalg.norm(d1_), 1e-15))
    belief_ov_l = np.array(belief_ov_l)
    z_naive = (ov - naive_64.mean()) / naive_64.std()
    z_belief = (ov - belief_ov_l.mean()) / belief_ov_l.std()
    print(f'  {lbl}: overlap={ov:.4f}  naive_z={z_naive:+.2f}  belief_z={z_belief:+.2f}')

# ── Visualization ──
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

ax = axes[0]
ax.hist(naive_overlap, bins=50, density=True, alpha=0.5, color='gray', label='Naive null\n(random directions)')
ax.hist(belief_overlap, bins=50, density=True, alpha=0.5, color='steelblue', label='Belief null\n(random eta projections)')
ax.axvline(obs_overlap, color='red', lw=2.5, label=f'Observed={obs_overlap:.3f}')
ax.set_xlabel('Subspace overlap (1D onto 3D)')
ax.set_ylabel('Density')
ax.set_title(f'Naive null z={naive_z:+.2f} | Belief null z={belief_z:+.2f}')
ax.legend(fontsize=8)

ax = axes[1]
x_pos = [0, 1, 2]
ax.bar(x_pos, [obs_overlap, naive_overlap.mean(), belief_overlap.mean()],
       yerr=[0, naive_overlap.std(), belief_overlap.std()],
       color=['#E53935', 'gray', 'steelblue'], alpha=0.8, capsize=5)
ax.set_xticks(x_pos)
ax.set_xticklabels(['Observed\n(driver/trans)', 'Naive null\nmean', 'Belief null\nmean'])
ax.set_ylabel('Overlap')
ax.set_title('Observed vs Both Null Distributions')

plt.suptitle('Subspace Analysis: Two Null Distributions', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_subspace.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Save Checkpoint

In [ ]:
save_dir = 'composite_mess3_checkpoint'
os.makedirs(save_dir, exist_ok=True)

# ── 1. Model weights (skip if already saved / loaded from checkpoint) ──
weights_path = f'{save_dir}/model_weights.pt'
if not os.path.exists(weights_path):
    torch.save(model.state_dict(), weights_path)
    print('Saved model weights.')
else:
    print('Model weights already exist, skipping.')

# ── 2. Config ──
config_dict = {
    'process': 'composite_mess3',
    'x_a': X_A, 'a_a': A_A, 'x_b': X_B, 'a_b': A_B, 'epsilon': EPSILON,
    'n_ctx': N_CTX, 'd_model': D_MODEL, 'd_head': D_HEAD,
    'n_heads': getattr(tl_cfg, 'n_heads', N_HEADS),
    'n_layers': N_LAYERS, 'd_mlp': D_MLP, 'd_vocab': VOCAB_SIZE,
    'act_fn': 'relu', 'normalization_type': 'LN', 'seed': SEED,
    'num_steps_trained': NUM_STEPS,
    'final_loss': float(losses[-1]),
    'full_belief_mse_concat': float(results['full_belief']['Concat']['mse']),
    'full_belief_r2_concat': float(results['full_belief']['Concat']['r2']),
    'driver_r2_concat': float(results['driver']['Concat']['r2']),
    'transducer_r2_concat': float(results['transducer']['Concat']['r2']),
}
with open(f'{save_dir}/config.json', 'w') as f:
    json.dump(config_dict, f, indent=2)

# ── 3. Training history ──
np.savez_compressed(f'{save_dir}/training_history.npz',
    losses=np.array(losses),
    checkpoint_steps=np.array(sorted(checkpoint_results.keys())),
    checkpoint_mses=np.array([checkpoint_results[s]['paper_mse'] for s in sorted(checkpoint_results)]),
    checkpoint_r2s=np.array([checkpoint_results[s]['r2'] for s in sorted(checkpoint_results)]),
)

# ── 4. Per-layer probe results ──
probe_data = {}
for probe_name in results:
    for layer_label in results[probe_name]:
        probe_data[f'{probe_name}_{layer_label}_r2'] = results[probe_name][layer_label]['r2']
        probe_data[f'{probe_name}_{layer_label}_mse'] = results[probe_name][layer_label]['mse']

# ── 5. Timescale data (L3 + Concat) ──
ts_data = {}
for label in results_ts:
    r = results_ts[label]
    ts_data[f'{label}_driver_mse'] = r['driver_mse']
    ts_data[f'{label}_driver_r2'] = r['driver_r2']
    ts_data[f'{label}_trans_mse'] = r['trans_mse']
    ts_data[f'{label}_trans_r2'] = r['trans_r2']
ts_data['theta_std'] = results_ts['L3']['theta_std']

np.savez_compressed(f'{save_dir}/probe_results.npz', **probe_data, **ts_data)

# ── 6. Controls ──
controls_dict = {
    'full_mse': full_mse,
    'untrained_mse': untrained_mse,
    'untrained_r2': untrained_r2,
    'seq_cv_test_mean': float(np.mean(seq_cv_test)),
    'seq_cv_test_std': float(np.std(seq_cv_test)),
    'pt_cv_test_mean': float(np.mean(pt_cv_test)),
    'pt_cv_test_std': float(np.std(pt_cv_test)),
    'tsplit_mse_avg': tsplit_mse_avg,
    'tsplit_mse_forward': tsplit_mse,
    'tsplit_mse_reverse': tsplit_mse_rev,
    'shuffle_mse_mean': float(np.mean(shuffle_mses)),
    'shuffle_mse_std': float(np.std(shuffle_mses)),
}
with open(f'{save_dir}/controls.json', 'w') as f:
    json.dump(controls_dict, f, indent=2)

# ── 7. Subspace analysis ──
subspace_dict = {
    'obs_overlap': float(obs_overlap),
    'obs_cos': [float(c) for c in obs_cos],
    'naive_null_mean': float(naive_overlap.mean()),
    'naive_null_std': float(naive_overlap.std()),
    'naive_z': float(naive_z),
    'belief_null_mean': float(belief_overlap.mean()),
    'belief_null_std': float(belief_overlap.std()),
    'belief_z': float(belief_z),
}
with open(f'{save_dir}/subspace.json', 'w') as f:
    json.dump(subspace_dict, f, indent=2)

# ── 9. Package and download ──
print(f'\nContents of {save_dir}/:')
for root, dirs, files_list in os.walk(save_dir):
    for fn in sorted(files_list):
        fp = os.path.join(root, fn)
        print(f'  {os.path.relpath(fp, save_dir)}: {os.path.getsize(fp)/1024:.0f} KB')

with zipfile.ZipFile('composite_mess3_checkpoint.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files_list in os.walk(save_dir):
        for fn in files_list:
            fp = os.path.join(root, fn)
            zf.write(fp)
print(f'\ncomposite_mess3_checkpoint.zip: {os.path.getsize("composite_mess3_checkpoint.zip")/1024:.0f} KB')

try:
    from google.colab import files
    files.download('composite_mess3_checkpoint.zip')
    print('Download started.')
except ImportError:
    print('Not in Colab -- zip saved locally.')
